# Vinograd, Nair et al. (2024): network time constants of the excitatory model (Figure 4c-f)

Paper: Vinograd A, Nair A, et al. (2024), Nature 634, 910-918,
[doi:10.1038/s41586-024-07915-x](https://doi.org/10.1038/s41586-024-07915-x). Dataset of the attempt:
[DANDI:001037](https://dandiarchive.org/dandiset/001037/0.260609.1402), version 0.260609.1402; this
notebook uses none of its data. The claim it tests is a simulation that the claims file lists
among the paper's findings; it uses no recorded data. fig4hj-inhibition-slow-dense, the model with
inhibition, depends on fig4a-dense-x1-connectivity, which is blocked (no unitary stimulation
sessions), and is therefore blocked as well.

Methods, Mechanistic modelling: "a model population of n = 1,000 standard current-based leaky
integrate-and-fire neurons"; "the network time constant tau_n was derived as tau_s / (1 - lambda_max),
where lambda_max is the largest eigenvalue of the synaptic weight matrix W"; "a subnetwork of 200
neurons (20% of the network), designated the integration subnetwork ... with varying densities of
random connectivity"; "Weights of the overall network were sampled from a uniform distribution:
W_ij ~ U(0, 1/N), while weights of the subnetwork were sampled as W_ij ~ U(0, 1/N_p), where
N_p = 200." Legend of Fig. 4c: densities "in the range of 0 to 20%", tau_s "in the range of 0 to 20 s";
unstable networks have runaway excitation (lambda_max >= 1).

The Methods do not say whether the subnetwork's weights are added to the background weights or
replace them within the subnetwork; both readings are run. Density sigma is the probability that a
subnetwork connection exists. Three random networks per density (seeds 0 to 2, a standard choice), on a grid of 0.5%.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import linattr as L

started = time.time()
N, NP = 1000, 200
print(f"Network model; no data are read from dandiset {L.DANDISET} (version {L.VERSION}) here.")


def weights(sigma, reading, rng):
    W = rng.uniform(0, 1 / N, (N, N))
    sub = rng.uniform(0, 1 / NP, (NP, NP)) * (rng.random((NP, NP)) < sigma)
    if reading == "added":
        W[:NP, :NP] += sub
    else:
        W[:NP, :NP] = sub
    np.fill_diagonal(W, 0)
    return W


def lam_max(W):
    return float(np.max(np.real(np.linalg.eigvals(W))))

## Data check

There are no data here; the check is of the model: the largest eigenvalue of the background alone
(density 0) and of a full subnetwork, against their expected values (a dense matrix of U(0, 1/N)
weights has a largest eigenvalue near N times the mean weight, 0.5).

In [ ]:
rng = np.random.default_rng(0)
for reading in ["added", "replaced"]:
    print(f"{reading:8s}: lambda_max at density 0 = {lam_max(weights(0.0, reading, rng)):.3f}; "
          f"at density 100% = {lam_max(weights(1.0, reading, rng)):.3f}")
print(f"N = {N}, subnetwork N_p = {NP} ({NP / N:.0%} of the network)")

## fig4ce-excitatory-network

> In purely excitatory spiking networks, only relatively sparse integration subnetworks (around 8 to
> 12%) with slow synaptic time constants (around 20 s) give network time constants of 50 to 200 s;
> fast glutamatergic networks fail over the same densities.

In [ ]:
SIGMAS = np.round(np.arange(0, 0.2001, 0.005), 4)
LAM = {}
for reading in ["added", "replaced"]:
    LAM[reading] = np.array([[lam_max(weights(s, reading, np.random.default_rng(seed))) for seed in range(3)]
                             for s in SIGMAS])
for reading in ["added", "replaced"]:
    lam = LAM[reading].mean(1)
    stable = lam < 1
    tau20 = np.where(stable, 20 / (1 - lam), np.inf)
    inside = stable & (tau20 >= 45) & (tau20 <= 220)
    print(f"reading {reading}: lambda_max {lam.min():.3f} to {lam.max():.3f}; stable for densities up to "
          f"{100 * SIGMAS[stable].max():.2f}%")
    print(f"   tau_s = 20 s: tau_n in 45-220 s at densities {100 * SIGMAS[inside].min() if inside.any() else float('nan'):.2f}% "
          f"to {100 * SIGMAS[inside].max() if inside.any() else float('nan'):.2f}% "
          f"({inside.sum()} grid points); inside 7.2-13.2%: {bool(inside[(SIGMAS >= 0.072) & (SIGMAS <= 0.132)].any())}")
    for s in [0.08, 0.10, 0.12]:
        i = int(np.argmin(abs(SIGMAS - s)))
        print(f"      density {100 * SIGMAS[i]:.1f}%: lambda_max {lam[i]:.3f} (sd over seeds {LAM[reading][i].std():.3f}), "
              f"tau_n = {tau20[i]:.1f} s")
    for ts in [0.01, 0.1, 0.2, 0.3, 0.6]:
        tn = np.where(stable, ts / (1 - lam), np.inf)
        ok = stable & (tn >= 45)
        print(f"   tau_s = {ts} s: largest stable tau_n {tn[stable].max():.2f} s; densities giving 45 s or more: {int(ok.sum())}")
    # per seed, the stable network closest to instability
    lam_seed = LAM[reading]
    st = lam_seed < 1
    print(f"   per network (not averaged): largest stable lambda_max {lam_seed[st].max():.4f}, so tau_n at 0.6 s at most "
          f"{0.6 / (1 - lam_seed[st].max()):.1f} s")

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for k, reading in enumerate(["added", "replaced"]):
    lam = LAM[reading].mean(1)
    for ts in [10, 15, 20]:
        tn = np.where(lam < 1, ts / (1 - lam), np.nan)
        ax[k].plot(100 * SIGMAS, tn, label=f"tau_s {ts} s")
    ax[k].axhspan(50, 200, color="#fdb863", alpha=0.3)
    ax[k].set_ylim(0, 400); ax[k].set_xlabel("subnetwork density (%)"); ax[k].set_ylabel("tau_n (s)")
    ax[k].set_title(f"subnetwork weights {reading}"); ax[k].legend(fontsize=7)
plt.show()
print("Points above 400 s (off the axes) are not drawn; unstable densities are left blank.")
print(f"run time {time.time() - started:.0f} s")

**Result: blocked (methods underspecified).** With the weights the Methods give, the largest
eigenvalue stays between 0.48 and 0.52 at every subnetwork density from 0 to 20% under both readings,
so at tau_s = 20 s the network time constant is 39 to 41 s at every density (40.7 s at 8% and 41.1 s at
12% with the weights added), never in 45 to 220 s, and no network is unstable. Fast networks
(tau_s 0.6 s or less) reach at most 1.3 s, as the claim says. The paper's Fig. 4c shows unstable
networks (runaway excitation) within the same density range, which the stated weights cannot
produce, so the networks behind the figure were scaled differently from the Methods' description;
the scale is not given anywhere in the paper.

## Summary

| Claim | Outcome |
|---|---|
| fig4ce-excitatory-network | blocked: methods underspecified (run; results above) |
| fig4hj-inhibition-slow-dense | blocked: data missing (depends on fig4a-dense-x1-connectivity) |

The analysis follows the Methods' weight distributions under two readings of how the subnetwork's
weights enter the matrix; neither gives the instability the figure shows.